# Content removed 
Removed all the fluffy comment from here..
We now start using some API to get/parse the content from a website and create a summary of it with AI

In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv # standard library to manage env vars
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

THere we just load the key from the .env file and run some basic verification

In [3]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [ ]:
# This create an instance of an OPenAI object
#
# Note that your Open API credit will expire after 1 year!!
openai = OpenAI()
#
# this is the message for chat.completions api
message = "Hello, GPT! This is my first ever message to you! Hi!"
# the call has a standard format  for modesl and messages:
# model : string for the model
# messages: list of dictionaries
#           each dictionary will have 2 keys: role and content
#           more on role later (user or system and they are related to the types of prompts)
response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role":"user", "content":message}])
# here we print the first choices and the content of the message 
print(response.choices[0].message.content)

#
# below is the Model response


Hi there! Nice to meet you, and welcome to the chat.

I can help with lots of things: explain concepts, answer questions, write or edit text, brainstorm ideas, code, translate, summarize, plan projects, practice a language, and more. What would you like to do today?

If you’re not sure yet, here are some quick ideas:
- Ask me to explain something you’re curious about
- Tell me a project or assignment, and I’ll help you with it
- I can draft or polish a piece of writing
- I can brainstorm ideas or plan steps for a goal
- We can practice a language or have a casual chat about hobbies
- Quick fun fact, riddle, or short story

Tell me a bit about your interests or what you want to achieve, and we’ll go from there!


##  First project - scraping a website

In [7]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [9]:
# Let's try one out. Change the website and add print statements to follow along.

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
Fo

## Types of prompts

**A system prompt** that tells them what task they are performing and what tone they should use (set the scene)

**A user prompt** -- the conversation starter that they should reply to

In [21]:
# Define our system prompt

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown"

In [18]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt_prefix = f"""Here are the content of the website: {website.title}
    Provide a short sumary of this website in mardown.
    If it includes news or announcementsm then summarize them too.

    """


    return user_prompt_prefix + website.text

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [17]:
#
# Here are the 2 types of roles/messages
# note that the original system message had 'useful' instead of 'snarky' and the tone of the answer was different
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [16]:
# Once again we call OpenAI with system and user messages:

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
print(response.choices[0].message.content)

Oh, such a classic! 2 + 2 is, unsurprisingly, 4. Shocking, right?


## And now let's build useful messages for GPT-4o-mini, using a function

In [19]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [20]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'Here are the content of the website: Home - Edward Donner\n    Provide a short sumary of this website in mardown.\n    If it includes news or announcementsm then summarize them too.\n\n    Skip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a 

## Time to bring it together - the API for OpenAI is very simple!

In [22]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [23]:
summarize("https://edwarddonner.com")

"# Summary of Edward Donner's Website\n\nEdward Donner is a passionate coder and experimenter with large language models (LLMs). He is co-founder and CTO of the AI startup Nebula.io and was previously founder and CEO of the AI startup untapt, acquired in 2021. Before that, he was a Managing Director at JPMorgan. He is enthusiastic about sharing knowledge on LLMs and has created popular Udemy courses with over 900,000 enrollments globally.\n\nHis interests also include amateur electronic music production and engaging with tech communities like Hacker News.\n\n## Key Features on the Website\n- **Curriculum:** A set of AI and LLM-focused courses that have become top-rated and widely enrolled.\n- **C4 and Outsmart:** Special features including Outsmart, an arena where LLMs compete in diplomacy and strategy.\n- **Digital Avatar:** An interactive chat with Edward’s digital avatar.\n- **Posts and Resources:** Various posts and resources related to AI, coding, and agent development.\n\n## Rece

In [ ]:
# Markdown and Display come from the initial imports
# from IPython.display import Markdown, display
#
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [25]:
display_summary("https://edwarddonner.com")

# Summary of Edward Donner's Website

Edward Donner is a software engineer and AI enthusiast who specializes in working with large language models (LLMs). He is the co-founder and CTO of the AI startup Nebula.io and was previously the founder and CEO of AI startup untapt, which was acquired in 2021. Edward also has experience as a Managing Director at JPMorgan.

He enjoys writing code, experimenting with LLMs, producing amateur electronic music, and engaging with tech discussions on Hacker News. Edward is passionate about teaching and has created several popular Udemy courses on AI, with over 900,000 enrollments from 194 countries.

## Main Features
- Personal background and professional experience
- Large curriculum of AI courses available
- A digital avatar for interactive chat
- An arena called "Outsmart" where LLMs compete in diplomacy and strategy

## Recent News / Announcements
- **February 17, 2026:** AI Coder: Vibe Coder to Agentic Engineer – RESOURCES  
- **January 4, 2026:** AI Builder with n8n – Create Agents and Voice Agents – RESOURCES  
- **September 15, 2025:** AI Engineering MLOps Track – Deploy AI to Production – RESOURCES  
- **May 28, 2025:** Guidance on which order to take the AI courses

## Contact & Follow
- Email: ed [at] edwarddonner [dot] com  
- Social: LinkedIn, Twitter, Facebook

This site is a resource hub for those interested in AI development, LLM experimentation, and related educational content.

In [20]:
display_summary("https://anthropic.com")

# Anthropic Overview

The Anthropic website primarily focuses on their AI product, **Claude**, and various applications designed to enhance human productivity and ensure safety in AI development. Here are the key highlights:

## Main Features:
- **Claude Models**: The site showcases various models, including **Claude Opus 4**, **Claude Sonnet 4**, and **Claude Haiku 3.5**, emphasizing their capabilities such as coding and AI agent functionalities.
- **API Integration**: Developers are encouraged to build custom applications using Claude through comprehensive API documentation and resources.

## News & Announcements:
- **ISO 42001 Certification**: Anthropic has announced achieving ISO 42001 certification, highlighting their commitment to safety in AI.
- **Latest Models**: Claude Opus 4 and Sonnet 4 are introduced as the most powerful models yet, capable of handling complex tasks and enhancing user productivity.

## Research & Safety:
- Anthropic prioritizes safety and responsible AI development, engaging in research that contemplates the societal impacts of AI. Key domains include:
  - **Core Views on AI Safety**: The company promotes the idea of developing AI with human benefits in mind.
  - **Responsible Scaling Policy**: This outlines Anthropic's approach to scaling AI technologies responsibly.

## Educational Resources:
- **Anthropic Academy**: An initiative to provide training and resources for users looking to effectively utilize Claude in various applications.

Overall, Anthropic's website positions the company as a leader in safe AI development with a focus on collaboration, productivity, and responsible innovation.

In [31]:
import json 
d = json.loads((response.to_json()))
print(f'Keys in the response message: {d.keys()}')
for k,v in d.items():
    if k not in ['choices', 'usage']:
        print(f' --- {k} : {v}')
    if k in ['usage']:
        print(' --- usage:')
        for usage_k,usage_v in v.items():
            print(f'      --- {usage_k} : {usage_v}')
    if k == 'choices':
        print(f' --- choices: {len(d['choices'])} ')
        for item in v:
            print(f'       - keys: {item.keys()}')           


Keys in the response message: dict_keys(['id', 'choices', 'created', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage'])
 --- id : chatcmpl-E8uXpqxc07hwDwpTmm4EOcW6M1Ha9
 --- choices: 1 
       - keys: dict_keys(['finish_reason', 'index', 'logprobs', 'message'])
 --- created : 1785792689
 --- model : gpt-4.1-nano-2025-04-14
 --- object : chat.completion
 --- service_tier : default
 --- system_fingerprint : fp_2ca6d3eb51
 --- usage:
      --- completion_tokens : 24
      --- prompt_tokens : 26
      --- total_tokens : 50
      --- completion_tokens_details : {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}
      --- prompt_tokens_details : {'audio_tokens': 0, 'cached_tokens': 0}


In [32]:
from pprint import pprint
import json
print(response.to_json())


{
  "id": "chatcmpl-E8uXpqxc07hwDwpTmm4EOcW6M1Ha9",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Oh, such a classic! 2 + 2 is, unsurprisingly, 4. Shocking, right?",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1785792689,
  "model": "gpt-4.1-nano-2025-04-14",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_2ca6d3eb51",
  "usage": {
    "completion_tokens": 24,
    "prompt_tokens": 26,
    "total_tokens": 50,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}
